# Do simple trading strategies survive out-of-sample?

**Question:** Two classic rule-based strategies look profitable in a backtest. Do they still beat buy-and-hold when they are tested on data they were not tuned on, and after transaction costs?

1. **Moving-average crossover on Apple (AAPL):** long when the short moving average is above the long one, short otherwise.
2. **Linear regression on lagged EUR/USD returns:** predict tomorrow's return or direction from the last two daily returns.

**Method**
- Daily adjusted prices from Yahoo Finance (`yfinance`), 2010–2025.
- Parameters and models are fitted on a **training period (2010–2017)** and evaluated on an untouched **test period (2018–2025)**.
- Positions are traded on the previous day's signal, so there is no look-ahead bias.
- Transaction costs are subtracted every time the position changes.

*The strategies are based on examples in Yves Hilpisch, *Python for Finance* (2nd ed.), chapter 15. The out-of-sample tests, costs and performance metrics are my own extension.*

In [ ]:
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

plt.style.use("seaborn-v0_8")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.float_format", "{:.4f}".format)

## Settings

In [ ]:
START = "2010-01-01"
END = "2025-12-31"
SPLIT_DATE = "2018-01-01"   # training data before this date, test data from it

COST_EQUITY = 0.0010        # 0.10% of the position traded (a flip from long to short trades 2 units)
COST_FX = 0.0002            # 0.02% for EUR/USD, a far more liquid market

TRADING_DAYS = 252
DATA_DIR = Path("data")     # downloaded prices are cached here (not committed to GitHub)

## Helper functions

In [ ]:
def load_prices(ticker: str) -> pd.Series:
    """Daily adjusted close prices from Yahoo Finance, cached as a CSV in data/."""
    DATA_DIR.mkdir(exist_ok=True)
    path = DATA_DIR / f"{ticker.replace('=', '_')}.csv"
    if path.exists():
        prices = pd.read_csv(path, index_col=0, parse_dates=True).iloc[:, 0]
    else:
        import yfinance as yf
        raw = yf.download(ticker, start=START, end=END, auto_adjust=True, progress=False)
        if raw.empty:
            raise RuntimeError(f"No data returned for {ticker}. Check the ticker and your connection.")
        prices = raw["Close"].squeeze()
        prices.to_csv(path)
    prices.name = ticker
    return prices.dropna()


def performance(log_returns: pd.Series) -> pd.Series:
    """Key statistics for a series of daily log returns."""
    equity = np.exp(log_returns.cumsum())
    years = len(log_returns) / TRADING_DAYS
    ann_vol = log_returns.std() * np.sqrt(TRADING_DAYS)
    return pd.Series({
        "Total return": equity.iloc[-1] - 1,
        "Annual return": equity.iloc[-1] ** (1 / years) - 1,
        "Annual volatility": ann_vol,
        "Sharpe ratio (rf = 0)": log_returns.mean() * TRADING_DAYS / ann_vol,
        "Max drawdown": (equity / equity.cummax() - 1).min(),
    })


def apply_costs(position: pd.Series, gross: pd.Series, cost: float) -> pd.Series:
    """Subtract costs each day the held position changes (|change| = units traded)."""
    turnover = position.diff().abs().fillna(0)
    return gross - turnover * cost


def train_test(df: pd.DataFrame):
    """Split a time-indexed frame at SPLIT_DATE."""
    return df[df.index < SPLIT_DATE], df[df.index >= SPLIT_DATE]

## Part 1 – Moving-average crossover on AAPL

The signal is computed at each day's close. The position is held **from the next day**, which is why the position is shifted by one day before it is multiplied by the market return.

In [ ]:
def sma_backtest(prices: pd.Series, short: int, long: int, cost: float = COST_EQUITY) -> pd.DataFrame:
    df = pd.DataFrame({"price": prices})
    df["sma_short"] = prices.rolling(short).mean()
    df["sma_long"] = prices.rolling(long).mean()
    df = df.dropna()

    df["signal"] = np.where(df["sma_short"] > df["sma_long"], 1, -1)
    df["held"] = df["signal"].shift(1)                     # trade on yesterday's signal
    df["market"] = np.log(df["price"] / df["price"].shift(1))
    df["gross"] = df["held"] * df["market"]
    df = df.dropna()
    df["net"] = apply_costs(df["held"], df["gross"], cost)
    return df


aapl = load_prices("AAPL")
base = sma_backtest(aapl, 42, 252)

ax = base[["price", "sma_short", "sma_long"]].rename(columns={
    "price": "Price", "sma_short": "42-day average", "sma_long": "252-day average"
}).plot(title="AAPL with 42- and 252-day moving averages")
base["held"].plot(ax=ax, secondary_y=True, style="--", alpha=0.4, color="grey")
ax.right_ax.set_ylabel("Position (+1 long / -1 short, dashed)")
ax.set_xlabel("");

### Choosing the windows on the training period only

The original exercise picks the best window pair on the full sample, which uses information that would not have been available at the time. Here the grid search only sees 2010–2017.

In [ ]:
shorts = range(20, 61, 4)
longs = range(180, 281, 10)

rows = []
for s, l in product(shorts, longs):
    train, _ = train_test(sma_backtest(aapl, s, l))
    rows.append({
        "short": s,
        "long": l,
        "market": np.exp(train["market"].sum()),
        "strategy (net)": np.exp(train["net"].sum()),
    })

grid = pd.DataFrame(rows)
grid["outperformance"] = grid["strategy (net)"] - grid["market"]
grid.sort_values("outperformance", ascending=False).head(7)

In [ ]:
heat = grid.pivot(index="short", columns="long", values="outperformance")

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(heat, cmap="RdBu", aspect="auto", origin="lower",
               vmin=-abs(heat).max().max(), vmax=abs(heat).max().max())
ax.set_xticks(range(len(heat.columns)), heat.columns)
ax.set_yticks(range(len(heat.index)), heat.index)
ax.set_xlabel("Long window (days)")
ax.set_ylabel("Short window (days)")
ax.set_title("Training period 2010–2017: net strategy minus buy-and-hold (growth multiple)")
ax.grid(False)
fig.colorbar(im, ax=ax);

A robust strategy would show a broad area of similar results. A single bright cell surrounded by weak ones suggests the "best" pair is mostly noise.

### Testing on 2018–2025

In [ ]:
best = grid.loc[grid["outperformance"].idxmax()]
best_short, best_long = int(best["short"]), int(best["long"])
print(f"Best training-period windows: {best_short}/{best_long}")

_, test_base = train_test(base)
_, test_best = train_test(sma_backtest(aapl, best_short, best_long))

aapl_results = pd.DataFrame({
    "Buy and hold": performance(test_best["market"]),
    "SMA 42/252 (net)": performance(test_base["net"]),
    f"SMA {best_short}/{best_long} gross": performance(test_best["gross"]),
    f"SMA {best_short}/{best_long} net": performance(test_best["net"]),
}).T
aapl_results["Trades"] = [0, test_base["held"].diff().abs().gt(0).sum(),
                          test_best["held"].diff().abs().gt(0).sum(),
                          test_best["held"].diff().abs().gt(0).sum()]
aapl_results

In [ ]:
equity = pd.DataFrame({
    "Buy and hold": test_best["market"],
    "SMA 42/252 (net)": test_base["net"],
    f"SMA {best_short}/{best_long} (net)": test_best["net"],
}).dropna().cumsum().apply(np.exp)

ax = equity.plot(title="AAPL test period: growth of 1 USD")
ax.set_xlabel("");

## Part 2 – Predicting EUR/USD from lagged returns

Two linear regressions use the last two daily log returns as features:
- **Model 1** predicts the size of tomorrow's return.
- **Model 2** predicts its direction (+1 / −1).

The position is +1 if the prediction is positive and −1 otherwise. The features are already lagged, so the prediction for day *t* only uses returns up to day *t − 1*. The models are fitted on 2010–2017 and evaluated on 2018–2025.

In [ ]:
eur = load_prices("EURUSD=X")

fx = pd.DataFrame({"returns": np.log(eur / eur.shift(1))}).dropna()
fx["direction"] = np.sign(fx["returns"]).astype(int)   # sign first, then int

LAGS = 2
lag_cols = [f"lag_{i}" for i in range(1, LAGS + 1)]
for i, col in enumerate(lag_cols, start=1):
    fx[col] = fx["returns"].shift(i)
fx = fx.dropna()

print(fx["direction"].value_counts())
fx.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fx["returns"].hist(bins=50, ax=axes[0])
axes[0].set_title("Distribution of daily EUR/USD log returns")

sc = axes[1].scatter(fx["lag_1"], fx["lag_2"], c=fx["returns"], cmap="coolwarm", s=8)
axes[1].axvline(0, c="k", ls="--", lw=0.8)
axes[1].axhline(0, c="k", ls="--", lw=0.8)
axes[1].set_xlabel("Return yesterday (lag 1)")
axes[1].set_ylabel("Return two days ago (lag 2)")
axes[1].set_title("Lagged returns, coloured by today's return")
fig.colorbar(sc, ax=axes[1]);

If the colours in the scatter plot look randomly mixed, past returns carry little information about the next one. That is what we would expect in an efficient FX market.

In [ ]:
fx_train, fx_test = train_test(fx)

models = {"OLS on returns": "returns", "OLS on direction": "direction"}
rows, test_net = [], {"Buy and hold": fx_test["returns"]}

for name, target in models.items():
    model = LinearRegression().fit(fx_train[lag_cols], fx_train[target])
    for period, data in [("train", fx_train), ("test", fx_test)]:
        position = pd.Series(np.where(model.predict(data[lag_cols]) > 0, 1, -1), index=data.index)
        net = apply_costs(position, position * data["returns"], COST_FX)
        moves = data["direction"] != 0
        stats = performance(net)
        stats["Hit ratio"] = (position[moves] == data.loc[moves, "direction"]).mean()
        stats["Trades"] = position.diff().abs().gt(0).sum()
        rows.append(pd.concat([pd.Series({"Model": name, "Period": period}), stats]))
        if period == "test":
            test_net[name] = net

for period, data in [("train", fx_train), ("test", fx_test)]:
    stats = performance(data["returns"])
    stats["Trades"] = 0
    rows.append(pd.concat([pd.Series({"Model": "Buy and hold", "Period": period}), stats]))

fx_results = pd.DataFrame(rows).set_index(["Model", "Period"])
fx_results["Trades"] = fx_results["Trades"].astype(int)
fx_results

In [ ]:
ax = pd.DataFrame(test_net).cumsum().apply(np.exp).plot(title="EUR/USD test period: growth of 1 unit (net of costs)")
ax.set_xlabel("");

A hit ratio close to 50% on the test period means the model is not much better than a coin flip. A clear drop from the training to the test period is a sign of overfitting.

## Conclusions

*Fill in after running the notebook with real data. Keep each point to one or two sentences with a number.*

- **AAPL crossover:** [Did the best training-period windows beat buy-and-hold in 2018–2025 after costs? Compare Sharpe ratios and drawdowns.]
- **Parameter stability:** [Was the heatmap smooth or noisy? What does that say about the "best" pair?]
- **EUR/USD regressions:** [Test-period hit ratios and net returns vs the training period.]
- **Takeaway:** [e.g. why in-sample backtests overstate performance and what you would test next.]

## Limitations and next steps

- Only one stock and one currency pair, so the results may not generalise.
- Flat costs. Real costs depend on spreads, market impact and borrowing costs for short positions.
- A single train/test split. A walk-forward test (re-fitting every year) would be more robust.
- The regressions use only two lags. Other features (volatility, interest-rate differentials) could be tested.